 ## PySpark Set Up

In [1]:
from pyspark.sql import SparkSession
import pandas as pd

# ── Start SparkSession ──
spark = SparkSession.builder \
    .appName("LogiScale_Phase1_DataCo") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()

# clean output
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("SparkSession started successfully.")

Spark version: 4.1.1
SparkSession started successfully.


 ## Load the  CSV file into PySpark

In [2]:
# Load the dataset
df_raw = spark.read.csv(
    "DataCoSupplyChainDataset.csv",
    header=True,                      # First row is treated as column names (not data).
    inferSchema=True,                 # Spark automatically detects data types: Without this, everything would be read as string.
    encoding="ISO-8859-1"             # Handles special characters
)

print(f"Raw row count  : {df_raw.count():,}")
print(f"Raw col count  : {len(df_raw.columns)}")
df_raw.printSchema()

Raw row count  : 180,519
Raw col count  : 53
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = t

## Drop Unnecessary Columns

In [3]:
# ── Drop columns NOT needed for this project ──
cols_to_drop = [
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Zipcode",
    "Customer City",
    "Customer State",
    "Product Description",
    "Product Image",
    "Product Status",
    "Order Zipcode",
    "Order Item Cardprod Id",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Total",
    "Product Card Id",
    "Product Category Id",
    "Order Customer Id",
    "Category Id",
    "Department Id"
]

#Spark DataFrames are immutable, so this does NOT modify df_raw. It creates a new one.
df_clean = df_raw.drop(*cols_to_drop)
 
print(f"\nColumns after drop : {len(df_clean.columns)}")
print("Remaining columns  :")
for c in df_clean.columns:
    print(f"  - {c}")


Columns after drop : 30
Remaining columns  :
  - Type
  - Days for shipping (real)
  - Days for shipment (scheduled)
  - Benefit per order
  - Sales per customer
  - Delivery Status
  - Late_delivery_risk
  - Category Name
  - Customer Country
  - Customer Id
  - Customer Segment
  - Department Name
  - Latitude
  - Longitude
  - Market
  - Order City
  - Order Country
  - order date (DateOrders)
  - Order Id
  - Order Item Id
  - Order Item Quantity
  - Sales
  - Order Profit Per Order
  - Order Region
  - Order State
  - Order Status
  - Product Name
  - Product Price
  - shipping date (DateOrders)
  - Shipping Mode


##  Remove Duplicates

In [4]:
# ── Count before ──
before = df_clean.count()

# ── Drop full duplicate rows ──
df_clean = df_clean.dropDuplicates()

# ── Also drop rows where Order Id is duplicate (keep first) ──
df_clean = df_clean.dropDuplicates(subset=["Order Id", "Order Item Id"])

after = df_clean.count()

print(f"\nRows before dedup : {before:,}")
print(f"Rows after dedup  : {after:,}")
print(f"Duplicates removed: {before - after:,}")


Rows before dedup : 180,519
Rows after dedup  : 180,519
Duplicates removed: 0


## Rename Columns to Match Project Names

In [5]:
# ── Rename columns to project-standard names ──
df = df_clean \
    .withColumnRenamed("Days for shipment (scheduled)", "eta_days") \
    .withColumnRenamed("Days for shipping (real)",      "ata_days") \
    .withColumnRenamed("Late_delivery_risk",            "late_risk") \
    .withColumnRenamed("Delivery Status",               "delivery_status") \
    .withColumnRenamed("Order Region",                  "route") \
    .withColumnRenamed("Order Country",                 "order_country") \
    .withColumnRenamed("Market",                        "market") \
    .withColumnRenamed("Department Name",               "warehouse_id") \
    .withColumnRenamed("Category Name",                 "category") \
    .withColumnRenamed("Product Name",                  "sku_name") \
    .withColumnRenamed("Order Item Quantity",           "units_sold") \
    .withColumnRenamed("Sales",                         "sales_amount") \
    .withColumnRenamed("Order Id",                      "order_id") \
    .withColumnRenamed("Order Item Id",                 "order_item_id") \
    .withColumnRenamed("Customer Id",                   "customer_id") \
    .withColumnRenamed("Customer Country",              "customer_country") \
    .withColumnRenamed("Customer Segment",              "customer_segment") \
    .withColumnRenamed("Order Status",                  "order_status") \
    .withColumnRenamed("Shipping Mode",                 "shipping_mode") \
    .withColumnRenamed("Order City",                    "order_city") \
    .withColumnRenamed("Order State",                   "order_state") \
    .withColumnRenamed("Benefit per order",             "benefit_per_order") \
    .withColumnRenamed("Sales per customer",            "sales_per_customer") \
    .withColumnRenamed("Order Profit Per Order",        "profit_per_order") \
    .withColumnRenamed("Product Price",                 "product_price") \
    .withColumnRenamed("Latitude",                      "latitude") \
    .withColumnRenamed("Longitude",                     "longitude") \
    .withColumnRenamed("order date (DateOrders)",       "order_date_raw") \
    .withColumnRenamed("shipping date (DateOrders)",    "ship_date_raw")

print("Columns renamed successfully.")
df.printSchema()

Columns renamed successfully.
root
 |-- Type: string (nullable = true)
 |-- ata_days: integer (nullable = true)
 |-- eta_days: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_risk: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- market: string (nullable = true)
 |-- order_city: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- order_date_raw: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- units_sold: integer (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- prof

 ## Fix Date Columns & Calculate Delay

In [8]:
from pyspark.sql.functions import col, when, to_timestamp

# ── Parse date columns (BEST VERSION) ──
df = df \
    .withColumn("order_date", to_timestamp(col("order_date_raw"), "M/d/yyyy H:mm")) \
    .withColumn("ship_date",  to_timestamp(col("ship_date_raw"),  "M/d/yyyy H:mm")) \
    .drop("order_date_raw", "ship_date_raw")

# ── Calculate delay in days ──
df = df.withColumn(
    "delay_days",
    col("ata_days") - col("eta_days")
)

# ── Add delivery flag ──
df = df.withColumn(
    "delivery_flag",
    when(col("delay_days") > 0,  "Late")
    .when(col("delay_days") < 0, "Early")
    .otherwise("On Time")
)

print("Date parsing and delay calculation done.")

df.select(
    "order_id", "route", "warehouse_id",
    "eta_days", "ata_days", "delay_days",
    "delivery_flag", "order_date", "ship_date"
).show(10)

Date parsing and delay calculation done.
+--------+---------------+------------+--------+--------+----------+-------------+-------------------+-------------------+
|order_id|          route|warehouse_id|eta_days|ata_days|delay_days|delivery_flag|         order_date|          ship_date|
+--------+---------------+------------+--------+--------+----------+-------------+-------------------+-------------------+
|       2|  South America|    Fan Shop|       4|       3|        -1|        Early|2015-01-01 00:21:00|2015-01-04 00:21:00|
|       2|  South America|     Apparel|       4|       3|        -1|        Early|2015-01-01 00:21:00|2015-01-04 00:21:00|
|       4|  South America|     Apparel|       4|       5|         1|         Late|2015-01-01 01:03:00|2015-01-06 01:03:00|
|       7|  South America|    Outdoors|       2|       3|         1|         Late|2015-01-01 02:06:00|2015-01-04 02:06:00|
|      12|Central America|    Footwear|       4|       3|        -1|        Early|2015-01-01 03:51

 ## Handle Nulls

In [9]:
from pyspark.sql.functions import isnan

# ── Count nulls in key columns ──
key_cols = ["eta_days", "ata_days", "route",
            "warehouse_id", "units_sold", "order_date"]

print("=== NULL CHECK ===")
for c in key_cols:
    null_count = df.filter(col(c).isNull()).count()
    status = "[OK]" if null_count == 0 else "[WARN]"
    print(f"  {status} {c}: {null_count:,} nulls")

# ── Drop rows where critical columns are null ──
df = df.dropna(subset=["eta_days", "ata_days", "route",
                        "warehouse_id", "order_date"])

print(f"\nFinal clean rows: {df.count():,}")

=== NULL CHECK ===
  [OK] eta_days: 0 nulls
  [OK] ata_days: 0 nulls
  [OK] route: 0 nulls
  [OK] warehouse_id: 0 nulls
  [OK] units_sold: 0 nulls
  [OK] order_date: 0 nulls

Final clean rows: 180,519


##  Performance Benchmark: Pandas vs PySpark

In [11]:
# ════════════════════════════════
#  PANDAS BENCHMARK
# ════════════════════════════════
import time
import pandas as pd

from pyspark.sql.functions import (
    col, when, avg,
    round as spark_round,
    to_timestamp
)
print("Running Pandas benchmark...")
t0 = time.time()

pdf = pd.read_csv(
    "DataCoSupplyChainDataset.csv",
    encoding="ISO-8859-1"
)
pdf["delay_days"] = (
    pdf["Days for shipping (real)"] -
    pdf["Days for shipment (scheduled)"]
)
pandas_result = pdf.groupby("Order Region")["delay_days"].mean()

pandas_time = round(time.time() - t0, 2)
print(f"Pandas  done → {pandas_time}s")

# ════════════════════════════════
#  PYSPARK BENCHMARK
# ════════════════════════════════
print("\nRunning PySpark benchmark...")
t0 = time.time()

spark_result = df.groupBy("route").agg(
    spark_round(avg("delay_days"), 2).alias("avg_delay_days")
)
spark_result.collect()   # collect() forces execution (Spark is lazy)

spark_time = round(time.time() - t0, 2)
print(f"PySpark done → {spark_time}s")

# ════════════════════════════════
#  PRINT RESULT
# ════════════════════════════════
print("\n" + "="*45)
print("   PERFORMANCE BENCHMARK RESULT")
print("="*45)
print(f"   Pandas  time : {pandas_time}s")
print(f"   PySpark time : {spark_time}s")
print(f"   Dataset rows : {df.count():,}")
print("="*45)
print("   At 10x data size — Pandas crashes,")
print("   PySpark scales without memory issues.")
print("="*45)

Running Pandas benchmark...
Pandas  done → 4.24s

Running PySpark benchmark...
PySpark done → 2.86s

   PERFORMANCE BENCHMARK RESULT
   Pandas  time : 4.24s
   PySpark time : 2.86s
   Dataset rows : 180,519
   At 10x data size — Pandas crashes,
   PySpark scales without memory issues.


## Save the Cleaned DataFrame 

In [12]:
# ── Save cleaned data as CSV for Phase 2 ──
df.toPandas().to_csv("dataco_cleaned.csv", index=False)
print("Cleaned file saved as: dataco_cleaned.csv")

# ── Quick final summary ──
print("\n=== PHASE 1 COMPLETE ===")
print(f"  Original rows   : {df_raw.count():,}")
print(f"  After cleaning  : {df.count():,}")
print(f"  Columns kept    : {len(df.columns)}")
print(f"  Key columns     : order_id, route, warehouse_id,")
print(f"                    eta_days, ata_days, delay_days,")
print(f"                    order_date, sku_name, units_sold")
print("========================")

Cleaned file saved as: dataco_cleaned.csv

=== PHASE 1 COMPLETE ===
  Original rows   : 180,519
  After cleaning  : 180,519
  Columns kept    : 32
  Key columns     : order_id, route, warehouse_id,
                    eta_days, ata_days, delay_days,
                    order_date, sku_name, units_sold
